In [0]:

%run ./00_imports


In [0]:

# ===============================================================
# La vue permet d'utiliser Spark SQL pour les agrégations Gold
# ===============================================================

df_silver = spark.table(TABLES["silver"])

# Enregistrement comme vue SQL temporaire
df_silver.createOrReplaceTempView("silver")

print(f"Table Silver chargée : {df_silver.count()} lignes")
print(f"Colonnes disponibles : {len(df_silver.columns)}")
print(f"\nColonnes :")
for col in df_silver.columns:
    print(f"  - {col}")

In [0]:

# ============================================================================
# Agrégation du taux de conformité par commune sur les 3 années
# ============================================================================

gold_conformite_commune = spark.sql("""
    SELECT
        code_commune,
        nom_commune,
        code_departement,
        COUNT(*)                                                    AS nb_analyses,
        SUM(CASE WHEN conforme = true  THEN 1 ELSE 0 END)          AS nb_conformes,
        SUM(CASE WHEN conforme = false THEN 1 ELSE 0 END)          AS nb_non_conformes,
        ROUND(
            100.0 * SUM(CASE WHEN conforme = true THEN 1 ELSE 0 END) 
            / COUNT(*), 2
        )                                                           AS taux_conformite_pct,
        MIN(date_prelevement)                                       AS premiere_analyse,
        MAX(date_prelevement)                                       AS derniere_analyse
    FROM silver
    WHERE code_commune IS NOT NULL
    GROUP BY code_commune, nom_commune, code_departement
    ORDER BY taux_conformite_pct ASC
""")

# Écriture en Gold
gold_conformite_commune.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLES["gold_commune"])

nb = spark.table(TABLES["gold_commune"]).count()
print(f"Table Gold écrite : {TABLES['gold_commune']}")
print(f"Communes traitées : {nb}")

display(spark.table(TABLES["gold_commune"]).limit(10))

In [0]:

# ==================================================================
# Taux de conformité mensuel par paramètre pour suivre les tendances
# ==================================================================

gold_evolution = spark.sql("""
    SELECT
        code_parametre,
        libelle_parametre,
        YEAR(date_prelevement)                                      AS annee,
        MONTH(date_prelevement)                                     AS mois,
        COUNT(*)                                                    AS nb_analyses,
        SUM(CASE WHEN conforme = true  THEN 1 ELSE 0 END)          AS nb_conformes,
        ROUND(
            100.0 * SUM(CASE WHEN conforme = true THEN 1 ELSE 0 END)
            / COUNT(*), 2
        )                                                           AS taux_conformite_pct,
        ROUND(AVG(valeur_numerique), 4)                             AS valeur_moyenne,
        ROUND(MAX(valeur_numerique), 4)                             AS valeur_max
    FROM silver
    WHERE date_prelevement IS NOT NULL
    AND conforme IS NOT NULL
    AND libelle_parametre NOT LIKE '%PRES/ABS%'
    AND libelle_parametre NOT LIKE '%CALCOCARBONIQUE%'
    AND libelle_parametre NOT LIKE '%SOUS ACCRÉDITATION%'
    AND libelle_parametre NOT LIKE '%PRÉLÈVEMENT%'
    AND libelle_parametre NOT LIKE '%(QUALITATIF)%'
    AND libelle_parametre NOT LIKE '%(O/N)%'
    GROUP BY code_parametre, libelle_parametre,
             YEAR(date_prelevement), MONTH(date_prelevement)
    ORDER BY annee, mois, taux_conformite_pct ASC
""")

gold_evolution.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLES["gold_evolution"])

nb = spark.table(TABLES["gold_evolution"]).count()
print(f"Table Gold écrite : {TABLES['gold_evolution']}")
print(f"Lignes produites  : {nb}")

display(spark.table(TABLES["gold_evolution"]).limit(10))

In [0]:


# ============================================================================
# Taux de conformité agrégé par département et par année pour visualisation
# ============================================================================

gold_carte = spark.sql("""
    SELECT
        code_departement,
        _annee                                                      AS annee,
        COUNT(*)                                                    AS nb_analyses,
        SUM(CASE WHEN conforme = true  THEN 1 ELSE 0 END)          AS nb_conformes,
        SUM(CASE WHEN conforme = false THEN 1 ELSE 0 END)          AS nb_non_conformes,
        ROUND(
            100.0 * SUM(CASE WHEN conforme = true THEN 1 ELSE 0 END)
            / COUNT(*), 2
        )                                                           AS taux_conformite_pct,
        COUNT(DISTINCT code_commune)                                AS nb_communes,
        COUNT(DISTINCT code_parametre)                              AS nb_parametres_analyses
    FROM silver
    WHERE code_departement IS NOT NULL
    AND conforme IS NOT NULL
    AND libelle_parametre NOT LIKE '%(QUALITATIF)%'
    AND libelle_parametre NOT LIKE '%(O/N)%'
    AND libelle_parametre NOT LIKE '%PRES/ABS%'
    AND libelle_parametre NOT LIKE '%CALCOCARBONIQUE%'
    AND libelle_parametre NOT LIKE '%SOUS ACCRÉDITATION%'
    AND libelle_parametre NOT LIKE '%PRÉLÈVEMENT%'
    GROUP BY code_departement, _annee
    ORDER BY taux_conformite_pct ASC
""")

gold_carte.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLES["gold_carte"])

nb = spark.table(TABLES["gold_carte"]).count()
print(f"Table Gold écrite : {TABLES['gold_carte']}")
print(f"Départements x années : {nb}")

display(spark.table(TABLES["gold_carte"]).limit(10))

In [0]:

# ============================================================================
# Classement par taux de conformité avec un minimum de 50 analyses
# ============================================================================

gold_top = spark.sql("""
    WITH conformite_commune AS (
        SELECT
            code_commune,
            nom_commune,
            code_departement,
            COUNT(*)                                                AS nb_analyses,
            ROUND(
                100.0 * SUM(CASE WHEN conforme = true THEN 1 ELSE 0 END)
                / COUNT(*), 2
            )                                                       AS taux_conformite_pct
        FROM silver
        WHERE code_commune IS NOT NULL
        AND conforme IS NOT NULL
        AND libelle_parametre NOT LIKE '%(QUALITATIF)%'
        AND libelle_parametre NOT LIKE '%(O/N)%'
        AND libelle_parametre NOT LIKE '%PRES/ABS%'
        AND libelle_parametre NOT LIKE '%CALCOCARBONIQUE%'
        AND libelle_parametre NOT LIKE '%SOUS ACCRÉDITATION%'
        AND libelle_parametre NOT LIKE '%PRÉLÈVEMENT%'
        GROUP BY code_commune, nom_commune, code_departement
        HAVING COUNT(*) >= 50
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (ORDER BY taux_conformite_pct ASC)  AS rang_moins_conforme,
            RANK() OVER (ORDER BY taux_conformite_pct DESC) AS rang_plus_conforme
        FROM conformite_commune
    )
    SELECT *,
        CASE
            WHEN rang_plus_conforme  <= 10 THEN 'Top 10 conforme'
            WHEN rang_moins_conforme <= 10 THEN 'Top 10 non conforme'
        END AS categorie
    FROM ranked
    WHERE rang_plus_conforme <= 10
    OR rang_moins_conforme   <= 10
    ORDER BY taux_conformite_pct ASC
""")

gold_top.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLES["gold_top"])

nb = spark.table(TABLES["gold_top"]).count()
print(f"Table Gold écrite : {TABLES['gold_top']}")
print(f"Communes dans le classement : {nb}")

display(spark.table(TABLES["gold_top"]).limit(20))

In [0]:

# ============================================================================
# Basée sur conclusionprel (PLV) — plus fiable que qualitparam par paramètre
# ============================================================================

gold_nonconf = spark.sql("""
    SELECT
        code_parametre,
        libelle_parametre,
        _annee                                                       AS annee,
        COUNT(*)                                                     AS nb_analyses,
        SUM(CASE WHEN conforme = false THEN 1 ELSE 0 END)            AS nb_non_conformes,
        ROUND(
            100.0 * SUM(CASE WHEN conforme = false THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0), 4
        )                                                            AS taux_non_conformite_pct,
        COUNT(DISTINCT code_commune)                                 AS nb_communes_concernees,
        COUNT(DISTINCT code_departement)                             AS nb_departements_concernes
    FROM silver
    WHERE conforme = false
    GROUP BY code_parametre, libelle_parametre, _annee
    ORDER BY nb_non_conformes DESC
""")

gold_nonconf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(TABLES["gold_nonconf"])

nb = spark.table(TABLES["gold_nonconf"]).count()
print(f"Table Gold écrite : {TABLES['gold_nonconf']}")
print(f"Combinaisons paramètre x année non-conformes : {nb}")

display(spark.table(TABLES["gold_nonconf"]).limit(15))